# Coarse answer + matcher support: three-set gate analysis (iteration 26)

Reads `evidence_<set>.csv` (from `Coarse_Evidence.ipynb`: 6 queries per photo, M=5 places, RANSAC inliers per place, no DINO) and `queries_<set>.csv` (coarse z, from `Coarse_Confidence.ipynb`).

Per photo: the **answer** is the coarse first place of the query with the highest coarse z. **sup** = the most RANSAC inliers any query gave to a place within 1 sigma of the answer (counted below the bar of 8 too). **skipped** = no query reached 8 inliers anywhere (the pipeline would not answer). A fallback answers a skipped photo with the coarse answer when sup >= k.

Kill rule written before the run: 0 answers beyond 8 sigma on all three sets with the same k, and (new set, above 800 m) at least 40 % of the skipped photos answered.

In [ ]:
import numpy as np, pandas as pd
R = "../results/coarse_evidence/"; C = "../results/coarse_confidence/"
def pts(e): return np.where(e<=1,10,np.where(e<=2,6,np.where(e<=4,3,np.where(e<=8,0,-20))))
def photo_table(tag, evfile):
    ev = pd.read_csv(R+evfile); q = pd.read_csv(C+"queries_%s.csv"%tag)
    m = ev.merge(q[["photo","q","z1"]], on=["photo","q"]); rows = []
    for p, g in m.groupby("photo"):
        b = g.loc[g.z1.idxmax()]; sup = 0
        for k in range(5):
            near = np.hypot(g["x%d"%k]-b.x0, g["y%d"%k]-b.y0)/b.sigma_m <= 1.0
            if near.any(): sup = max(sup, g["inl%d"%k][near].max())
        rows.append(dict(photo=p, height=b.height, z=b.z1, sup=sup, best_inl=g.best_inl.max(), err=b.err0))
    t = pd.DataFrame(rows); t["pts"] = pts(t.err.values); return t
sets = {"new (332)": photo_table("highalt_dev332","evidence_highalt_dev332.csv"),
        "official (500)": photo_table("official500","evidence_official500.csv"),
        "NUDT (300)": photo_table("nudt300","evidence_nudt300.csv")}
for n, s in sets.items():
    print(n, "always-answer pts/photo", round(s.pts.mean(),2), "wrong", int((s.err>8).sum()), " wrong coarse answers with sup>0:", int(((s.err>8)&(s.sup>0)).sum()))

## 1. Fallback on photos the bar skips, by k

In [ ]:
table = []
for n, s in sets.items():
    sk = s[s.best_inl < 8]
    for k in [1,2,3,4,5,6,7]:
        sel = sk[sk.sup >= k]
        table.append(dict(set=n, skipped=len(sk), k=k, answered=len(sel), wrong=int((sel.err>8).sum()), within2=int((sel.err<=2).sum()), points=int(sel.pts.sum()), pts_per_photo=round(sel.pts.sum()/len(s),2)))
table = pd.DataFrame(table); print(table.to_string(index=False))

## 2. Is k chosen on one set safe on the others? (leave one set out)

Smallest k with 0 wrong on the two other sets, then its result on the held-out set.

In [ ]:
names = list(sets)
for held in names:
    others = [n for n in names if n != held]
    for k in range(1, 8):
        if all(table[(table.set==o)&(table.k==k)].wrong.iloc[0]==0 for o in others): break
    r = table[(table.set==held)&(table.k==k)].iloc[0]
    print("held out", held, "-> k =", k, "(clean on", others, ") held-out answered", r.answered, "wrong", r.wrong, "points", r.points)

## 3. New set above 800 m only

In [ ]:
s = sets["new (332)"]; hi = s[s.height >= 800]; sk = hi[hi.best_inl < 8]
print("above 800 m:", len(hi), "skipped by bar:", len(sk), "coarse-right2 among skipped:", int((sk.err<=2).sum()))
for k in [1,3,4,5,6]:
    sel = sk[sk.sup>=k]; print("k", k, "answered", len(sel), "of", len(sk), "wrong", int((sel.err>8).sum()), "points", int(sel.pts.sum()))
print("share of skipped photos answered at k=4:", round(len(sk[sk.sup>=4])/len(sk),2), "(kill rule asks for >= 0.40)")